# Deep Sets Autoencoder Training Experiments

Interactive notebook for:
- Training and evaluating the Deep Sets model
- Hyperparameter tuning
- Threshold optimization
- Error analysis
- Embedding visualization

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Import project modules
from dataset.hackathon import HackathonDataset
from dataset.collate import collate_fn
from models.deep_sets_autoencoder import DeepSetsAutoencoder
from config.train_config import get_default_config, get_fast_experiment_config
from utils.features import compute_class_weights, FeatureNormalizer
from utils.losses import create_loss_function
from utils.evaluation import (
    evaluate_predictions,
    find_optimal_threshold,
    compute_per_operation_metrics,
    analyze_errors
)

%matplotlib inline
sns.set_style("whitegrid")

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

OSError: Could not find kaggle.json. Make sure it's located in C:\Users\tobia\.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/

## 1. Setup and Configuration

In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load configuration
config = get_fast_experiment_config()  # Use fast config for quick experiments

# Display configuration
print(f"\nModel Configuration:")
print(f"  Embedding dim: {config.model.embedding_dim}")
print(f"  Hidden dim: {config.model.hidden_dim}")
print(f"  Pooling: {config.model.pooling_type}")
print(f"  Use attention: {config.model.use_attention}")

print(f"\nTraining Configuration:")
print(f"  Batch size: {config.training.batch_size}")
print(f"  Epochs: {config.training.num_epochs}")
print(f"  Learning rate: {config.training.learning_rate}")
print(f"  Use focal loss: {config.loss.use_focal_loss}")

## 2. Load Datasets

In [ ]:
# Load datasets
print("Loading datasets...")

train_dataset = HackathonDataset(
    root=config.data_root,
    split="train",
    download=False,
    sampling_strategy=config.sampling.train_sampling_strategies,
    seed=config.seed
)

val_dataset = HackathonDataset(
    root=config.data_root,
    split="val",
    download=False,
    sampling_strategy=config.sampling.val_sampling_strategies,
    seed=config.seed
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Validation dataset: {len(val_dataset)} samples")

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.training.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.training.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 3. Explore Sample Batch

In [ ]:
# Get a sample batch
sample_batch = next(iter(train_loader))

print("Batch structure:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: {type(value)}")

# Analyze X and Y
X = sample_batch["X"]
Y = sample_batch["Y"]
context = sample_batch["context"]
context_mask = sample_batch["context_mask"]

print(f"\nX (visible operations + room cluster):")
print(f"  Shape: {X.shape}")
print(f"  Mean ops per room: {X[:, :388].sum(dim=1).mean():.2f}")

print(f"\nY (masked operations):")
print(f"  Shape: {Y.shape}")
print(f"  Mean masked ops: {Y.sum(dim=1).mean():.2f}")

print(f"\nContext (other rooms):")
print(f"  Shape: {context.shape}")
print(f"  Mean valid rooms: {context_mask.sum(dim=1).float().mean():.2f}")

## 4. Initialize Model

In [ ]:
# Create model
model = DeepSetsAutoencoder(
    num_clusters=config.model.num_clusters,
    embedding_dim=config.model.embedding_dim,
    hidden_dim=config.model.hidden_dim,
    num_companies=config.model.num_companies,
    num_rooms=config.model.num_rooms,
    pooling_type=config.model.pooling_type,
    use_attention=config.model.use_attention,
    dropout=config.model.dropout,
    use_auxiliary_head=config.model.use_auxiliary_head
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created successfully!")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

## 5. Class Weights Analysis

In [ ]:
# Compute class weights
try:
    class_weights = compute_class_weights()
    
    print(f"Class weights computed:")
    print(f"  Min: {class_weights.min():.4f}")
    print(f"  Max: {class_weights.max():.4f}")
    print(f"  Mean: {class_weights.mean():.4f}")
    print(f"  Median: {class_weights.median():.4f}")
    
    # Plot distribution
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.hist(class_weights.numpy(), bins=50, edgecolor='black')
    plt.xlabel('Class Weight')
    plt.ylabel('Frequency')
    plt.title('Distribution of Class Weights')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(sorted(class_weights.numpy(), reverse=True))
    plt.xlabel('Operation Rank')
    plt.ylabel('Class Weight')
    plt.title('Class Weights (Sorted)')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Show top and bottom operations by weight
    top_k = 10
    top_indices = class_weights.topk(top_k).indices
    bottom_indices = class_weights.topk(top_k, largest=False).indices
    
    print(f"\nTop {top_k} rarest operations (highest weights):")
    for i, idx in enumerate(top_indices.tolist()):
        print(f"  {i+1}. Op {idx}: weight = {class_weights[idx]:.4f}")
    
    print(f"\nTop {top_k} most common operations (lowest weights):")
    for i, idx in enumerate(bottom_indices.tolist()):
        print(f"  {i+1}. Op {idx}: weight = {class_weights[idx]:.4f}")
        
except Exception as e:
    print(f"Could not compute class weights: {e}")
    class_weights = None

## 6. Training Loop (Simplified)

In [ ]:
# Create loss function and optimizer
loss_fn = create_loss_function(config.loss, class_weights, device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.training.learning_rate,
    weight_decay=config.training.weight_decay
)

# Training history
history = {
    'train_loss': [],
    'train_main_loss': [],
    'train_aux_loss': [],
    'val_loss': [],
    'val_f1': [],
    'val_room_score': []
}

print("Setup complete! Ready to train.")
print(f"\nNote: For full training, use train.py script.")
print(f"This notebook is for experimentation with smaller datasets/epochs.")

## 7. Threshold Optimization

In [ ]:
# Note: This requires a trained model
# For demonstration, we'll show the process

# To run after training:
# 1. Get predictions on validation set
# 2. Find optimal threshold using find_optimal_threshold()
# 3. Visualize performance vs threshold

print("Threshold optimization process:")
print("""  
1. Run inference on validation set
2. Test thresholds from 0.1 to 0.9
3. Compute room_score for each threshold
4. Select threshold with best score
5. Optionally: per-operation thresholds
""")

# Example code (requires trained model):
# model.eval()
# with torch.no_grad():
#     val_preds = []
#     val_targets = []
#     for batch in val_loader:
#         # ... run inference ...
#         val_preds.append(predictions)
#         val_targets.append(batch["Y"])
#     
#     val_preds = torch.cat(val_preds)
#     val_targets = torch.cat(val_targets)
#     
#     best_thresh, best_score = find_optimal_threshold(
#         val_preds, val_targets, metric="room_score", num_steps=41
#     )
#     print(f"Best threshold: {best_thresh:.3f} (score: {best_score:.4f})")

## 8. Error Analysis

In [ ]:
# After training, analyze errors
print("Error analysis workflow:")
print("""
1. Compute per-operation precision/recall
2. Identify operations with high FP/FN rates
3. Analyze error patterns by room type
4. Check co-occurrence of errors
5. Visualize confusion patterns
""")

# Example: analyze_errors(val_preds, val_targets, threshold=0.5, top_k=20)

## 9. Embedding Visualization

In [ ]:
# Visualize learned operation embeddings
print("Embedding visualization:")
print("""
1. Extract embedding layer weights
2. Apply dimensionality reduction (PCA/t-SNE/UMAP)
3. Plot embeddings in 2D
4. Color by operation frequency or room type
5. Identify clusters of related operations
""")

# Example:
# embeddings = model.operation_embedding.weight.data.cpu().numpy()
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# embeddings_2d = pca.fit_transform(embeddings)
# plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.5)
# plt.title("Operation Embeddings (PCA)")

## 10. Hyperparameter Tuning Suggestions

In [ ]:
print("Hyperparameters to tune:")
print("""
Model Architecture:
  - embedding_dim: [64, 128, 256]
  - hidden_dim: [128, 256, 512]
  - pooling_type: ['mean', 'sum', 'max']
  - use_attention: [True, False]
  - dropout: [0.1, 0.2, 0.3]

Training:
  - learning_rate: [1e-4, 5e-4, 1e-3]
  - batch_size: [16, 32, 64]
  - weight_decay: [1e-5, 1e-4, 1e-3]

Loss:
  - use_focal_loss: [True, False]
  - focal_gamma: [1.0, 2.0, 3.0]
  - auxiliary_loss_weight: [0.05, 0.1, 0.2]

Sampling:
  - sample_pct: [0.3, 0.5, 0.7]
  - use_balanced_data: [True, False]
""")

## Summary

This notebook provides an interactive environment for:
- Quick experiments with the Deep Sets model
- Analyzing data characteristics
- Visualizing embeddings and errors
- Tuning hyperparameters

For full-scale training, use the `train.py` script with appropriate configuration.